# S0 · Safe source intake and conversion audit

This notebook audits the safe conversion checkpoint. The original pickle is deliberately excluded from the final package. The full conversion implementation is in `src/profile_and_prepare.py` and the restricted unpickler audit is preserved under `audit/`.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import numpy as np
import psutil
from IPython.display import display, Image

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"
AUDIT = ROOT / "audit"
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
RUN_LOG = LOGS / "run_log.txt"

def checkpoint(label, *frames, started=None):
    elapsed = time.time() - started if started is not None else 0.0
    shapes = [getattr(x, "shape", None) for x in frames]
    nulls = []
    for frame in frames:
        if hasattr(frame, "isna"):
            nulls.append(round(float(frame.isna().mean(numeric_only=False).mean()), 6))
    rss = psutil.Process().memory_info().rss / 1024**3
    line = f"{label} | shapes={shapes} | mean_null_rates={nulls} | rss_gib={rss:.3f} | elapsed_s={elapsed:.3f}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")

t0 = time.time()
print(f"Audit root: {ROOT}")
checkpoint("setup", started=t0)

Audit root: <PROJECT_ROOT>/5_最终交付包/_rebuild/MA-Hackathon-Final-2026-09-04
setup | shapes=[] | mean_null_rates=[] | rss_gib=0.113 | elapsed_s=0.000


In [2]:
t0 = time.time()
profile = json.loads((AUDIT / "raw_conversion_profile.json").read_text())
schema = pd.read_csv(AUDIT / "raw_schema.csv")
flow = pd.read_csv(OUTPUTS / "analysis_sample_flow.csv")
display(pd.DataFrame([profile]))
display(schema)
display(flow)
checkpoint("S0 source manifests", schema, flow, started=t0)

,input_file,input_size_bytes,row_count,column_count,columns,arrow_schema,sampled_python_types,part_count,chunk_rows,elapsed_seconds,restricted_unpickler,raw_personal_data_in_final_package
0,Kiva_Loans.pkl,1597994508,1453846,27,"[activity, borrowerCount, city, country_iso, c...",activity: string\nborrowerCount: int64\ncity: ...,"{'activity': ['str'], 'borrowerCount': ['int']...",30,50000,30.191,True,False


,column_name,column_type,null,key,default,extra
0,activity,VARCHAR,YES,NaN,NaN,NaN
1,borrowerCount,BIGINT,YES,NaN,NaN,NaN
2,city,VARCHAR,YES,NaN,NaN,NaN
3,country_iso,VARCHAR,YES,NaN,NaN,NaN
4,country_latitude,DOUBLE,YES,NaN,NaN,NaN
5,country_longitude,DOUBLE,YES,NaN,NaN,NaN
6,country_name,VARCHAR,YES,NaN,NaN,NaN
7,country_ppp,DOUBLE,YES,NaN,NaN,NaN
8,description,VARCHAR,YES,NaN,NaN,NaN
9,disbursalDate,VARCHAR,YES,NaN,NaN,NaN


,step,loans,excluded_at_step,reason
0,Raw official extract,1453846,0,Source total
1,Unique loan IDs,1453846,0,No duplicate IDs
2,Valid nonnegative duration,1453840,6,raisedDate earlier than fundraisingDate
3,72-hour label eligible,1453346,494,At least 72 hours of calendar follow-up; also ...
4,"Training 2016-2024, valid duration",1316678,0,Learned preprocessing and estimation period
5,"2025 holdout, valid duration",137162,0,Untouched out-of-time evaluation period


S0 source manifests | shapes=[(28, 6), (6, 4)] | mean_null_rates=[0.5, 0.0] | rss_gib=0.114 | elapsed_s=0.012


In [3]:
t0 = time.time()
opcode = json.loads((AUDIT / "pickle_opcode_audit.json").read_text())
assert opcode["dangerous_opcode_count"] == 0
assert int(flow.iloc[0]["loans"]) == 1_453_846
assert not (ROOT / "data" / "Kiva_Loans.pkl").exists()
print("PASS: restricted opcode audit, expected source row count, and raw-data exclusion")
display(pd.DataFrame([opcode]))
checkpoint("S0 safety assertions", flow, started=t0)

PASS: restricted opcode audit, expected source row count, and raw-data exclusion


,input_file,input_size_bytes,scan_elapsed_seconds,stop_position,trailing_bytes_after_stop,dangerous_opcode_count,dangerous_opcodes_first_100,max_text_length,max_bytes_length,opcode_counts,safe_for_restricted_primitive_deserialization,privacy_note
0,Kiva_Loans.pkl,1597994508,98.642,1597994507,0,0,[],17834,0,"{'APPENDS': 1454, 'BINFLOAT': 5842290, 'BINGET...",True,Arguments were not logged because the pickle c...


S0 safety assertions | shapes=[(6, 4)] | mean_null_rates=[0.0] | rss_gib=0.114 | elapsed_s=0.003
